# LFM2 — Tool Calling

## Imports

In [1]:
import ast
import inspect
import json
import keyword
import re
from pprint import pprint

import torch
import transformers
from transformers.utils import get_json_schema

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Tokenizer

In [2]:
MODEL_ID = "LiquidAI/LFM2-700M"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Architecture: ['Lfm2ForCausalLM']
Parameters: 742,489,344
Device: mps:0


## Define Tools

In [3]:
def get_weather(location: str):
    """
    Get the current weather for a location.

    Args:
        location: The city to get the weather for, e.g. "San Francisco"
    """
    return {"location": location, "temperature": 68, "unit": "F", "conditions": "sunny"}


def get_stock_price(ticker: str):
    """
    Get the latest closing price for a stock ticker.

    Args:
        ticker: The stock ticker symbol, e.g. "AAPL"
    """
    return {"ticker": ticker, "price": 294.38, "currency": "USD"}


def convert_currency(amount: float, from_currency: str, to_currency: str):
    """
    Convert an amount of money from one currency to another.

    Args:
        amount: The amount of money to convert
        from_currency: The currency code to convert from, e.g. "USD"
        to_currency: The currency code to convert to, e.g. "EUR"
    """
    rate = 0.88
    return {"amount": round(amount * rate, 2), "currency": to_currency, "rate": rate}


TOOL_REGISTRY = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
}

In [4]:
tools = [get_json_schema(fn)["function"] for fn in TOOL_REGISTRY.values()]
pprint(tools, sort_dicts=False, width=120)

[{'name': 'get_weather',
  'description': 'Get the current weather for a location.',
  'parameters': {'type': 'object',
                 'properties': {'location': {'type': 'string',
                                             'description': 'The city to get the weather for, e.g. "San Francisco"'}},
                 'required': ['location']}},
 {'name': 'get_stock_price',
  'description': 'Get the latest closing price for a stock ticker.',
  'parameters': {'type': 'object',
                 'properties': {'ticker': {'type': 'string', 'description': 'The stock ticker symbol, e.g. "AAPL"'}},
                 'required': ['ticker']}},
 {'name': 'convert_currency',
  'description': 'Convert an amount of money from one currency to another.',
  'parameters': {'type': 'object',
                 'properties': {'amount': {'type': 'number', 'description': 'The amount of money to convert'},
                                'from_currency': {'type': 'string',
                                      

LFM2 emits tool calls as a Python-style call list between `<|tool_call_start|>` and `<|tool_call_end|>`, so the response needs to be decoded with the special tokens intact and then parsed.

In [5]:
TOOL_CALL_PATTERN = re.compile(r"<\|tool_call_start\|>(.*?)(?:<\|tool_call_end\|>|$)", re.DOTALL)
RESERVED_ARGUMENT = re.compile(rf"\b({'|'.join(keyword.kwlist)})\s*=")


def decode_response(outputs, input_len):
    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=False)
    return text.removesuffix("<|im_end|>").strip()


def flatten_calls(node):
    if isinstance(node, (ast.List, ast.Tuple, ast.Set)):
        for element in node.elts:
            yield from flatten_calls(element)
    elif isinstance(node, ast.Subscript):
        yield from flatten_calls(node.value)
        yield from flatten_calls(node.slice)
    else:
        yield node


def argument_value(node):
    try:
        return ast.literal_eval(node)
    except ValueError:
        return ast.unparse(node)


def build_tool_call(node):
    if isinstance(node, ast.Dict):
        call = ast.literal_eval(node)
        name, arguments = call["name"], call.get("arguments", {})
    else:
        name = node.func.attr if isinstance(node.func, ast.Attribute) else node.func.id
        arguments = {keyword_arg.arg: argument_value(keyword_arg.value) for keyword_arg in node.keywords}
        if node.args:
            parameters = list(inspect.signature(TOOL_REGISTRY[name]).parameters)
            arguments.update(zip(parameters, (argument_value(arg) for arg in node.args)))

    accepted = inspect.signature(TOOL_REGISTRY[name]).parameters
    return {"name": name, "arguments": {k: v for k, v in arguments.items() if k in accepted}}


def parse_tool_calls(text):
    tool_calls = []
    for block in TOOL_CALL_PATTERN.findall(text):
        block = RESERVED_ARGUMENT.sub(r"\1_=", block.strip())
        for statement in ast.parse(block).body:
            tool_calls.extend(build_tool_call(node) for node in flatten_calls(statement.value))
    return tool_calls

## Single Tool Call

In [6]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>system
List of tools: <|tool_list_start|>[{"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city to get the weather for, e.g. \"San Francisco\""}}, "required": ["location"]}}, {"name": "get_stock_price", "description": "Get the latest closing price for a stock ticker.", "parameters": {"type": "object", "properties": {"ticker": {"type": "string", "description": "The stock ticker symbol, e.g. \"AAPL\""}}, "required": ["ticker"]}}, {"name": "convert_currency", "description": "Convert an amount of money from one currency to another.", "parameters": {"type": "object", "properties": {"amount": {"type": "number", "description": "The amount of money to convert"}, "from_currency": {"type": "string", "description": "The currency code to convert from, e.g. \"USD\""}, "to_currency": {"type": "string", "description": "The currency code to conv

In [7]:
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|tool_call_start|>[get_weather(location="San Francisco")]<|tool_call_end|>


In [8]:
tool_calls = parse_tool_calls(response)
pprint(tool_calls, sort_dicts=False, width=120)

[{'name': 'get_weather', 'arguments': {'location': 'San Francisco'}}]


In [9]:
messages.append({"role": "assistant", "content": response})

for tool_call in tool_calls:
    result = TOOL_REGISTRY[tool_call["name"]](**tool_call["arguments"])
    print(f"{tool_call['name']}({tool_call['arguments']}) -> {result}")
    messages.append({"role": "tool", "content": json.dumps(result)})

pprint(messages, sort_dicts=False, width=120)

get_weather({'location': 'San Francisco'}) -> {'location': 'San Francisco', 'temperature': 68, 'unit': 'F', 'conditions': 'sunny'}
[{'role': 'user', 'content': "What's the weather in San Francisco?"},
 {'role': 'assistant', 'content': '<|tool_call_start|>[get_weather(location="San Francisco")]<|tool_call_end|>'},
 {'role': 'tool', 'content': '{"location": "San Francisco", "temperature": 68, "unit": "F", "conditions": "sunny"}'}]


In [10]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>system
List of tools: <|tool_list_start|>[{"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city to get the weather for, e.g. \"San Francisco\""}}, "required": ["location"]}}, {"name": "get_stock_price", "description": "Get the latest closing price for a stock ticker.", "parameters": {"type": "object", "properties": {"ticker": {"type": "string", "description": "The stock ticker symbol, e.g. \"AAPL\""}}, "required": ["ticker"]}}, {"name": "convert_currency", "description": "Convert an amount of money from one currency to another.", "parameters": {"type": "object", "properties": {"amount": {"type": "number", "description": "The amount of money to convert"}, "from_currency": {"type": "string", "description": "The currency code to convert from, e.g. \"USD\""}, "to_currency": {"type": "string", "description": "The currency code to conv

In [11]:
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

The current weather in San Francisco is 68 degrees Fahrenheit and sunny.


## Multiple Tool Calls

In [12]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco and the stock price of AAPL?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

<|tool_call_start|>[get_weather(location="San Francisco"), get_stock_price("AAPL")]<|tool_call_end|>


In [13]:
tool_calls = parse_tool_calls(response)
pprint(tool_calls, sort_dicts=False, width=120)

[{'name': 'get_weather', 'arguments': {'location': 'San Francisco'}},
 {'name': 'get_stock_price', 'arguments': {'ticker': 'AAPL'}}]


In [14]:
messages.append({"role": "assistant", "content": response})

for tool_call in tool_calls:
    result = TOOL_REGISTRY[tool_call["name"]](**tool_call["arguments"])
    print(f"{tool_call['name']}({tool_call['arguments']}) -> {result}")
    messages.append({"role": "tool", "content": json.dumps(result)})

pprint(messages, sort_dicts=False, width=120)

get_weather({'location': 'San Francisco'}) -> {'location': 'San Francisco', 'temperature': 68, 'unit': 'F', 'conditions': 'sunny'}
get_stock_price({'ticker': 'AAPL'}) -> {'ticker': 'AAPL', 'price': 294.38, 'currency': 'USD'}
[{'role': 'user', 'content': "What's the weather in San Francisco and the stock price of AAPL?"},
 {'role': 'assistant',
  'content': '<|tool_call_start|>[get_weather(location="San Francisco"), get_stock_price("AAPL")]<|tool_call_end|>'},
 {'role': 'tool', 'content': '{"location": "San Francisco", "temperature": 68, "unit": "F", "conditions": "sunny"}'},
 {'role': 'tool', 'content': '{"ticker": "AAPL", "price": 294.38, "currency": "USD"}'}]


In [15]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

The current weather in San Francisco is 68 degrees Fahrenheit and sunny. The latest closing price for AAPL is $294.38.


## Sequential Tool Calls

Some requests need the result of one tool before the next can be called. The system prompt keeps the model from guessing an argument it cannot know yet.

In [16]:
messages = [
    {
        "role": "system",
        "content": (
            "You must call tools one at a time and wait for each result before deciding "
            "the next step. Never guess a tool argument that depends on a previous tool result."
        ),
    },
    {"role": "user", "content": "What's the stock price of AAPL in euros?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>system
You must call tools one at a time and wait for each result before deciding the next step. Never guess a tool argument that depends on a previous tool result.
List of tools: <|tool_list_start|>[{"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city to get the weather for, e.g. \"San Francisco\""}}, "required": ["location"]}}, {"name": "get_stock_price", "description": "Get the latest closing price for a stock ticker.", "parameters": {"type": "object", "properties": {"ticker": {"type": "string", "description": "The stock ticker symbol, e.g. \"AAPL\""}}, "required": ["ticker"]}}, {"name": "convert_currency", "description": "Convert an amount of money from one currency to another.", "parameters": {"type": "object", "properties": {"amount": {"type": "number", "description": "The amount of money to convert"}, "from_currency": {"t

In [17]:
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

<|tool_call_start|>[get_stock_price(ticker="AAPL", currency="EUR")]<|tool_call_end|>


In [18]:
tool_calls = parse_tool_calls(response)
messages.append({"role": "assistant", "content": response})

for tool_call in tool_calls:
    result = TOOL_REGISTRY[tool_call["name"]](**tool_call["arguments"])
    print(f"{tool_call['name']}({tool_call['arguments']}) -> {result}")
    messages.append({"role": "tool", "content": json.dumps(result)})

get_stock_price({'ticker': 'AAPL'}) -> {'ticker': 'AAPL', 'price': 294.38, 'currency': 'USD'}


In [19]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

The stock price of AAPL is 294.38 USD, which is equivalent to 294.38 EUR.


In [20]:
tool_calls = parse_tool_calls(response)
messages.append({"role": "assistant", "content": response})

for tool_call in tool_calls:
    result = TOOL_REGISTRY[tool_call["name"]](**tool_call["arguments"])
    print(f"{tool_call['name']}({tool_call['arguments']}) -> {result}")
    messages.append({"role": "tool", "content": json.dumps(result)})

In [21]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

The stock price of AAPL is 294.38 USD, which is equivalent to 294.38 EUR.


In [22]:
pprint(messages, sort_dicts=False, width=120)

[{'role': 'system',
  'content': 'You must call tools one at a time and wait for each result before deciding the next step. Never guess a '
             'tool argument that depends on a previous tool result.'},
 {'role': 'user', 'content': "What's the stock price of AAPL in euros?"},
 {'role': 'assistant',
  'content': '<|tool_call_start|>[get_stock_price(ticker="AAPL", currency="EUR")]<|tool_call_end|>'},
 {'role': 'tool', 'content': '{"ticker": "AAPL", "price": 294.38, "currency": "USD"}'},
 {'role': 'assistant', 'content': 'The stock price of AAPL is 294.38 USD, which is equivalent to 294.38 EUR.'}]
